# Lab 9 · Tools and ReAct: build an agent that acts, with the small model

**Time:** 60 min · **Work in teams of 3** · **You will:** give the small model **tools**, watch it **ask** for them, run the loop **by hand** (that loop is ReAct), then let LangChain's **`create_agent`** do the same in one call, read its **trace**, **measure** it on ten requests and **break it on purpose**.

**Why it matters:** an LLM only reads and writes text. A tool is how it changes the world (set the AC) or fetches what it does not know. Everything else today (LangGraph, memory, guardrails, evals) is built on this one idea: **the model asks, your code acts**.

**The model is Qwen2.5-3B running on your GPU.** There is no API and no key. `lc_slm.py` wraps it as a normal LangChain chat model, so the same LangChain code you will meet in any project works on it.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each run, then read the result.*

In [ ]:
import sys, os
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check
from lc_slm import SmallChatModel
import graph_slm
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage
import pandas as pd

model = SmallChatModel()                      # the small model, as a LangChain chat model
reply = model.invoke("Say hello in five words.")
print(type(reply).__name__, "->", reply.content)

## 9.1 Tools: a function plus a description
The home is a plain Python dictionary. A **tool** is a normal function with a decorator. The model never sees the code: it sees the **name**, the **docstring** and the **argument types**. So the docstring is the instruction.

In [ ]:
ROOMS = ["living room", "bedroom", "kitchen", "study", "balcony"]
HOME = {r: {"light": "off", "ac": None, "blinds": "closed"} for r in ROOMS}

def reset_home():
    for r in ROOMS:
        HOME[r].update(light="off", ac=None, blinds="closed")

def show_home():
    print({r: (v["light"], v["ac"], v["blinds"]) for r, v in HOME.items() if v != {"light": "off", "ac": None, "blinds": "closed"}} or "nothing changed")

@tool
def set_light(room: str, state: str) -> str:
    """Turn the light on or off in a room. room is one of: living room, bedroom, kitchen, study, balcony. state is 'on' or 'off'."""
    if room not in HOME:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    HOME[room]["light"] = state
    return f"The {room} light is {state}."

@tool
def set_ac(room: str, temp: int) -> str:
    """Set the air conditioner of a room to a temperature in degrees Celsius, from 16 to 30."""
    if room not in HOME:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    HOME[room]["ac"] = temp
    return f"The {room} AC is set to {temp} degrees."

@tool
def set_blinds(room: str, state: str) -> str:
    """Open or close the blinds in a room. state is 'open' or 'closed'."""
    if room not in HOME:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    HOME[room]["blinds"] = state
    return f"The {room} blinds are {state}."

TOOLS = [set_light, set_ac, set_blinds]
print("name       :", set_light.name)
print("description:", set_light.description)
print("arguments  :", set_light.args)

**Read it:** that is everything the model is told about `set_light`. If the description is vague, the model guesses. You will test that later in this lab.

## 9.2 The model asks for a tool; it does not run it
`bind_tools` hands the tool descriptions to the model. Watch what comes back: **no text, a tool call**.

In [ ]:
model_with_tools = model.bind_tools(TOOLS)
ask = model_with_tools.invoke("It is too dark in the kitchen.")
print("text    :", repr(ask.content))
print("tool call:", ask.tool_calls)
show_home()

The home did **not** change. The model only **asked**. Your code has to run the tool. Do it yourself, and send the result back as a **ToolMessage** so the model can write the final answer:

In [ ]:
call = ask.tool_calls[0]
result = set_light.invoke(call["args"])                    # your code runs the tool
print("tool result:", result)
show_home()

messages = [HumanMessage("It is too dark in the kitchen."), ask, ToolMessage(content=result, tool_call_id=call["id"])]
final = model_with_tools.invoke(messages)
print("final answer:", final.content)

In [ ]:
check(HOME["kitchen"]["light"] == "on" and not final.tool_calls,
      "The model asked, the code acted, the result went back, the model answered. That is one full tool call.",
      "the kitchen light should be on and the final reply should have no tool call.")

## 9.3 The loop by hand: this is ReAct
**ReAct** = **Re**asoning + **Act**ing. The model thinks, asks for a tool, reads the result, and thinks again, until it can answer. It is a loop of four lines. Read it, then run it:

In [ ]:
def react_by_hand(request, max_steps=4, verbose=True):
    messages = [HumanMessage(request)]
    for step in range(max_steps):
        ai = model_with_tools.invoke(messages)                # the model thinks and may ask for tools
        messages.append(ai)
        if not ai.tool_calls:                                 # no tool asked: this is the final answer
            if verbose: print(f"step {step + 1}: ANSWER  {ai.content}")
            return messages
        for c in ai.tool_calls:                               # run every tool it asked for
            tool_fn = {t.name: t for t in TOOLS}[c["name"]]
            obs = tool_fn.invoke(c["args"])
            messages.append(ToolMessage(content=obs, tool_call_id=c["id"]))
            if verbose: print(f"step {step + 1}: ACTION  {c['name']}({c['args']})  ->  OBSERVATION  {obs}")
    return messages

reset_home()
trace = react_by_hand("Make the bedroom 22 degrees and open its blinds.")
show_home()

### 🔧 Try it 9.1 — Your own request
**Predict first:** how many tool calls does *"turn off the living room light and set the study AC to 24"* need? One step or two?

🔧 **Try:** change the request below. Watch the steps. Is every call right?

In [ ]:
reset_home()
my_request = "Turn on the study light and set the study AC to 24."        # 🔧 your own request
trace = react_by_hand(my_request)
show_home()

## 9.4 The same loop in one call: `create_agent`
LangChain ships this loop as `create_agent`. Give it the model, the tools and a short system prompt. It returns an **agent**: you send messages in, you get the whole conversation back.

In [ ]:
agent = create_agent(model=model, tools=TOOLS,
                     system_prompt="You control a smart home. Use the tools to act, then confirm in one short sentence.")

reset_home()
out = agent.invoke({"messages": [{"role": "user", "content": "Make the bedroom 22 degrees and open its blinds."}]})
for m in out["messages"]:
    m.pretty_print()
show_home()

## 9.5 Reading a trace
`out["messages"]` is the **trace**. Read it top to bottom: **Human** → **AI asks** (tool calls, no text) → **Tool** (the observation) → **AI answers** (no tool calls). When an agent misbehaves, the trace shows exactly where.

In [ ]:
def trace_summary(out):
    """One line per step, so a trace is easy to read and to test."""
    lines = []
    for m in out["messages"]:
        kind = type(m).__name__.replace("Message", "")
        if kind == "AI" and m.tool_calls:
            lines.append("AI asks   " + ", ".join(f"{c['name']}({c['args']})" for c in m.tool_calls))
        elif kind == "AI":
            lines.append("AI answers " + m.content[:70])
        elif kind == "Tool":
            lines.append("Tool       " + m.content)
        else:
            lines.append("Human      " + m.content)
    return lines

def tool_calls_of(out):
    return [(c["name"], c["args"]) for m in out["messages"] if type(m).__name__ == "AIMessage" for c in m.tool_calls]

print("\n".join(trace_summary(out)))
print()
print("tool calls:", tool_calls_of(out))

In [ ]:
calls = tool_calls_of(out)
check(len(calls) >= 2 and {c[0] for c in calls} >= {"set_ac", "set_blinds"},
      "Two requests in one sentence, two tool calls, both in the trace.",
      "the agent should have called set_ac and set_blinds.")

## 9.6 Measure it: ten requests
One run proves nothing. We score the agent on ten requests. A request is **right** when the agent asked for the **right tool** with the **right room** (and value).

In [ ]:
CASES = [
    ("It is too dark in the kitchen.",                       "set_light",  {"room": "kitchen", "state": "on"}),
    ("Switch off the light in the bedroom.",                 "set_light",  {"room": "bedroom", "state": "off"}),
    ("Make the study 24 degrees.",                           "set_ac",     {"room": "study", "temp": 24}),
    ("Set the living room AC to 21.",                        "set_ac",     {"room": "living room", "temp": 21}),
    ("Open the blinds in the living room.",                  "set_blinds", {"room": "living room", "state": "open"}),
    ("Close the balcony blinds.",                            "set_blinds", {"room": "balcony", "state": "closed"}),
    # three that need a little inference: the user did not name the device or the value
    ("It is boiling in the study.",                          "set_ac",     {"room": "study"}),
    ("Too much sun in the living room, cover the windows.",  "set_blinds", {"room": "living room", "state": "closed"}),
    ("Lights out in the balcony, I am going to sleep.",      "set_light",  {"room": "balcony", "state": "off"}),
    ("Let some sun into the bedroom.",                       "set_blinds", {"room": "bedroom", "state": "open"}),
]

def same(got, want):
    return all(str(got.get(k)).lower() == str(v).lower() for k, v in want.items())

def run_cases(agent, cases=CASES, alias=None):
    """Run every case. `alias` maps our tool names to the names the agent knows (used in 9.7)."""
    alias = alias or {}
    rows = []
    for request, want_tool, want_args in cases:
        want_tool = alias.get(want_tool, want_tool)
        reset_home()
        out = agent.invoke({"messages": [{"role": "user", "content": request}]})
        calls = tool_calls_of(out)
        first = calls[0] if calls else (None, {})
        rows.append({"request": request, "tool": first[0], "args": first[1], "calls": len(calls),
                     "right tool": first[0] == want_tool, "right args": first[0] == want_tool and same(first[1], want_args)})
    return pd.DataFrame(rows)

results = run_cases(agent)
pd.set_option("display.max_colwidth", 60)
results[["request", "tool", "right tool", "right args", "calls"]]

In [ ]:
score_tool = results["right tool"].mean()
score_args = results["right args"].mean()
print(f"right tool: {score_tool:.0%}    right tool AND right arguments: {score_args:.0%}")
check(score_tool >= 0.6,
      f"The small model picks the right tool {score_tool:.0%} of the time and gets tool and arguments right {score_args:.0%}. Ten cases show a trend, not a guarantee: Lab 14 builds a proper test set.",
      "fewer than 6 of 10 requests chose the right tool: check that the GPU is used and the tool descriptions are clear.")

**Read it:** the first number is "did it pick the right tool"; the second adds "with the right values". The gap between them is where **invented or wrong arguments** hide. Look at the rows where **right args** is `False` and read the `args` column.

## 9.7 Break it on purpose
**A. Tools the model cannot tell apart.** The model reads only the description. So what if the **names** say nothing and the **descriptions** are lazy? Below, the same three functions are given the names `device_a`, `device_b`, `device_c`. In one agent they keep their clear descriptions, in the other every description is just *"Changes a device."*. **Predict:** how much worse is the second agent?

In [ ]:
from langchain_core.tools import StructuredTool

def anonymous_tools(descriptions):
    """The same three functions under neutral names, with the descriptions you choose."""
    return [StructuredTool.from_function(func=t.func, name=name, description=desc, args_schema=t.args_schema)
            for t, name, desc in zip(TOOLS, ["device_a", "device_b", "device_c"], descriptions)]

alias = {"set_light": "device_a", "set_ac": "device_b", "set_blinds": "device_c"}
SYSTEM = "You control a smart home. Use the tools to act, then confirm in one short sentence."
clear_agent = create_agent(model=model, tools=anonymous_tools([t.description for t in TOOLS]), system_prompt=SYSTEM)
vague_agent = create_agent(model=model, tools=anonymous_tools(["Changes a device."] * 3), system_prompt=SYSTEM)
clear = run_cases(clear_agent, alias=alias)
vague = run_cases(vague_agent, alias=alias)
print(f"neutral names, clear descriptions: tool {clear['right tool'].mean():.0%}, tool+args {clear['right args'].mean():.0%}")
print(f"neutral names, vague descriptions: tool {vague['right tool'].mean():.0%}, tool+args {vague['right args'].mean():.0%}")

In [ ]:
check(clear["right tool"].mean() > vague["right tool"].mean(),
      "With nothing but the description to go on, a clear description wins clearly. The docstring is the instruction: write it like one.",
      "the clear descriptions should beat the vague ones; look at the two scores above.")

**B. A request no tool can do.** The garage does not exist. Watch what the agent does: refuse, invent, or call a wrong tool?

In [ ]:
reset_home()
out = agent.invoke({"messages": [{"role": "user", "content": "Make the garage warmer."}]})
print("\n".join(trace_summary(out)))
show_home()

**C. A step limit.** An agent that loops would run for ever. LangGraph (which runs `create_agent` inside) stops it at a **recursion limit** and raises an error. Set a very low one and catch it:

In [ ]:
from langgraph.errors import GraphRecursionError
try:
    agent.invoke({"messages": [{"role": "user", "content": "Make the bedroom 22 degrees and open its blinds."}]},
                 config={"recursion_limit": 3})
    print("finished inside 3 steps")
except GraphRecursionError as e:
    print("Stopped by the step limit:", str(e)[:90], "...")

## 9.8 Your turn: write a tool
### 🔧 TODO 9 — Add the `ac_off` tool
The agent cannot switch an AC **off** (`set_ac` needs a temperature). Write the tool, add it to the agent and test it.

In [ ]:
@tool
def ac_off(room: str) -> str:
    """TODO: write a clear description for the model."""
    # TODO: switch the AC of the room off (HOME[room]["ac"] = None) and return a sentence
    return "TODO: not written yet"

agent2 = create_agent(model=model, tools=TOOLS + [ac_off],
                      system_prompt="You control a smart home. Use the tools to act, then confirm in one short sentence.")
reset_home(); HOME["kitchen"]["ac"] = 20
out2 = agent2.invoke({"messages": [{"role": "user", "content": "Turn off the kitchen AC."}]})
print("\n".join(trace_summary(out2)))

In [ ]:
check(HOME["kitchen"]["ac"] is None,
      "Your new tool was found by the model and switched the kitchen AC off. A tool needs a clear name, a clear description and a return value the model can read.",
      "the kitchen AC should be off: check the docstring of ac_off and that you used HOME[room]['ac'] = None.")

## 9.9 What you proved

In [ ]:
summary = pd.DataFrame([
    {"experiment": "clear descriptions, 10 requests", "right tool": f"{score_tool:.0%}", "right tool + args": f"{score_args:.0%}"},
    {"experiment": "neutral names, clear descriptions", "right tool": f"{clear['right tool'].mean():.0%}", "right tool + args": f"{clear['right args'].mean():.0%}"},
    {"experiment": "neutral names, vague descriptions", "right tool": f"{vague['right tool'].mean():.0%}", "right tool + args": f"{vague['right args'].mean():.0%}"},
])
summary

**In one line:** *an agent is a loop in which the model asks for tools and your code runs them. The small model does this well enough to be useful and badly enough to need what comes next: a graph around it (Labs 10 and 11), memory (Lab 12), guardrails (Lab 13) and evals (Lab 14).*

## Check your understanding
1. In section 9.2 the home did not change after the model "called" `set_light`. Who changes it, and why is that important?
2. What does the loop in `react_by_hand` do, and how does `create_agent` relate to it?
3. Why did a vaguer docstring change the result? What does that tell you about writing tools?
4. The agent met "Make the garage warmer". What would you want a production agent to do, and where would you put that rule?

*✍️ Your answer:* 

## Next
Lab 10 draws your agent as a graph and then builds your own graph with LangGraph: state, nodes, edges and a conditional edge, with the small model in the first node.